In [ ]:

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

class DroneHoverEnv(gym.Env):

    def __init__(self):
        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(1,),
            dtype=np.float32
        )
        self.observation_space = spaces.Box(
            low=np.array([-20, -5]),
            high=np.array([20, 5]),
            dtype=np.float32
        )
        self.target_height = 5
        self.gravity = -0.05

        self.max_steps = 200
    def reset(self, seed=None, options=None):
        self.height = np.random.uniform(0, 1)
        self.velocity = 0
        self.steps = 0
        state = np.array([self.height, self.velocity], dtype=np.float32)
        return state, {}

    def step(self, action):
        action = np.clip(action, -1, 1)
        thrust = action[0]
        self.velocity += thrust + self.gravity
        self.velocity = np.clip(self.velocity, -3, 3)
        self.height += self.velocity
        self.steps += 1
        distance = abs(self.height - self.target_height)
        reward = -distance
        done = False
        if self.height < 0 or self.height > 20:
            done = True
        if self.steps >= self.max_steps:
            done = True
        state = np.array([self.height, self.velocity], dtype=np.float32)

        return state, reward, done, False, {}
class Actor(nn.Module):

    def __init__(self, state_dim, action_dim):

        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.mean = nn.Linear(64, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))


    def forward(self, state):
        x = self.fc(state)

        mean = self.mean(x)
        std = torch.exp(self.log_std)

        return mean, std
class Critic(nn.Module):

    def __init__(self, state_dim):

        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        return self.net(state)

env = DroneHoverEnv()
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
actor = Actor(state_dim, action_dim)
critic = Critic(state_dim)
actor_optimizer = optim.Adam(actor.parameters(), lr=0.0005)
critic_optimizer = optim.Adam(critic.parameters(), lr=0.001)
gamma = 0.99
episodes = 400

for ep in range(episodes):

    state, _ = env.reset()
    state = torch.FloatTensor(state)

    done = False
    total_reward = 0

    while not done:
        mean, std = actor(state)

        dist = Normal(mean, std)

        action = dist.sample()
        log_prob = dist.log_prob(action)
        action_np = action.detach().numpy()
        next_state, reward, done, _, _ = env.step(action_np)
        next_state = torch.FloatTensor(next_state)
        value = critic(state)
        next_value = critic(next_state)

        target = reward + gamma * next_value * (1 - int(done))
        advantage = target - value
        critic_loss = advantage.pow(2).mean()

        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()


        actor_loss = -(log_prob * advantage.detach()).mean()

        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()
        state = next_state
        total_reward += reward


    print(f"Episode {ep} Reward: {total_reward:.2f}")

Episode 0 Reward: -142.54
Episode 1 Reward: -17.06
Episode 2 Reward: -5.46
Episode 3 Reward: -5.09
Episode 4 Reward: -10.80
Episode 5 Reward: -5.70
Episode 6 Reward: -15.10
Episode 7 Reward: -15.37
Episode 8 Reward: -5.48
Episode 9 Reward: -57.55
Episode 10 Reward: -57.68
Episode 11 Reward: -5.57
Episode 12 Reward: -68.62
Episode 13 Reward: -20.03
Episode 14 Reward: -18.50
Episode 15 Reward: -19.74
Episode 16 Reward: -55.82
Episode 17 Reward: -58.20
Episode 18 Reward: -67.71
Episode 19 Reward: -64.19
Episode 20 Reward: -58.68
Episode 21 Reward: -69.12
Episode 22 Reward: -18.71
Episode 23 Reward: -10.54
Episode 24 Reward: -10.53
Episode 25 Reward: -23.80
Episode 26 Reward: -10.25
Episode 27 Reward: -56.66
Episode 28 Reward: -64.92
Episode 29 Reward: -65.35
Episode 30 Reward: -55.02
Episode 31 Reward: -5.75
Episode 32 Reward: -17.32
Episode 33 Reward: -5.25
Episode 34 Reward: -5.38
Episode 35 Reward: -11.37
Episode 36 Reward: -67.73
Episode 37 Reward: -5.98
Episode 38 Reward: -10.33
Epis